# 04 · Outflow destinations (y0–y3)

Inputs: `utxo_life`, `address_key`, `address_stats`, `key_exposure`, `holder_snapshot`, `holder_snapshot_placebo`, `events`, `placebo_days` (02, 02b, 03)

Outputs: `holder_snapshot_pc`, `exchange_addresses_raw`, `exchange_cospend`, `exchange_keys`, `flow_universe`, `key_flags`, `flow_days`, `event_flow_coverage`, `placebo_draws_strict`, `flow_tx_inputs`, `flow_tx_outputs`, `dest_key_info`, `flow_key_tx`, `key_day_flows`

- Freeze date 2026-09-22: flows are tagged up to this day; later spends are treated as unspent.
- Change: an output to any input key of the transaction; not an outflow. With several input keys, each key's outflow = its share of input value × non-change outputs. Change sent to a new address counts as y1.
- Destination classes for non-change outputs, assigned in order:
  1. y2 exchange: receiving key in `exchange_keys`
  2. y0 exposed: P2PK, bare multisig, P2TR, or a receiving key exposed on or before the transfer day (address reuse)
  3. y1 loose: has a key_id and is not exposed at the time of the transfer
  4. other: non-standard outputs without a key_id

  y3 = y2 + y0 + y1 loose + other. y1 mid = y1 loose to a key first seen in this transaction. y1 strict = y1 loose to an address first seen in this transaction, hash-type (P2PKH/P2WPKH/P2SH/P2WSH) and unspent for the next 30 days.
- Risk-set exclusions: exchange keys, and keys with any address receiving more than 1,000 transfers over its lifetime (`key_flags.is_service`).

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

RISK_MIN_BTC       = 0.01    # risk set: minimum balance at t-1
RISK_LOOKBACK_DAYS = 365     # risk set: incoming transfer within 12 months (not applied to positive controls)
SERVICE_MAX_RECV   = 1000    # any address with more lifetime receipts -> service/custodian, excluded
MAIN_WIN   = (-14, 30)       # main window
WIDE_WIN   = (-60, 60)       # wider window for main events and positive controls, for pre-trend plots
DORMANT_DAYS = 30            # y1 strict: unspent for the next 30 days
EXPAND_COSPEND   = True      # expand exchange labels by one hop of common-input spending
COSPEND_MAX_IN   = 1000      # expansion uses only transactions with at most this many inputs
N_DRAWS, SEED    = 1000, 20260923
AUDIT_N          = 200       # size of the y1-strict audit sample

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import requests

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["utxo_life", "address_key", "address_stats", "key_exposure", "holder_snapshot",
          "holder_snapshot_placebo", "events", "placebo_days", "placebo_draws"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from earlier steps found. Spent in this Colab session: $%.2f / $%.0f (the ledger file lives in the current session only)" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    # BYTES columns cannot be clustered
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:                       # dates are always stored as DATE (joinable with the date columns of utxo_life)
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

SATS_MIN = int(round(RISK_MIN_BTC * 1e8))
FREEZE_DATE = "2026-09-22"   # freeze date = latest snapshot date (utxo_life has only part of the blocks of 09-23, which are not used)
DATA_END = pd.Timestamp(FREEZE_DATE)
_latest = pd.Timestamp(q(f"SELECT MAX(snap_date) AS d FROM `{DS}.holder_snapshot` WHERE event = 'latest'").d.iloc[0])
assert _latest == DATA_END, f"latest snapshot date {_latest.date()} differs from the freeze date {FREEZE_DATE}"
print("Freeze date (last data day):", DATA_END.date())

## Step 1 · Official exchange address lists

- OKX: address files of each release on the proof-of-reserves download page (https://www.okx.com/en-us/proof-of-reserves/download); union of the latest six releases.
- Binance: the four BTC cold-wallet addresses in the blog post *Our Commitment to Transparency* (2022-11-10), listed in the cell.
- Bitfinex: `wallets.txt` in the GitHub repository `bitfinexcom/pub` (reserves disclosure, 2022-11).

Parsing: in tabular files with a currency column, BTC rows only; in plain text, addresses on lines containing "BTC"/"bitcoin" and none of BCH/BSV/Cash/Gold (all address-like strings if the file has no currency labels). Only addresses seen on the Bitcoin chain enter `exchange_keys` (filtered on `address_key` in step 3). bech32 addresses are lower-cased.

Additional lists are read from `exchange_addresses_manual.csv` (columns address, exchange, source_url, source_date).

In [ ]:
EXCHANGE_SOURCES = [
    # (exchange, source date, url)
    ("OKX", "2026-07-07", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026070700_V3.zip"),
    ("OKX", "2026-06-19", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026061900_V3.zip"),
    ("OKX", "2026-05-07", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026050700_V4.zip"),
    ("OKX", "2026-04-20", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026042000_V3.zip"),
    ("OKX", "2026-03-03", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026030319_V3.zip"),
    ("OKX", "2026-02-04", "https://static.okx.com/cdn/okx/por/chain/por_csv_2026020419_V4.zip"),
    ("Bitfinex", "2022-11", "https://raw.githubusercontent.com/bitfinexcom/pub/main/wallets.txt"),
]
BINANCE_URL = "https://www.binance.com/en/blog/community/our-commitment-to-transparency-2895840147147652626"
INLINE_ADDRESSES = [   # (address, exchange, source_url, source_date)
    ("34xp4vRoCGJym3xR7yCVPFHoCNxv4Twseo", "Binance", BINANCE_URL, "2022-11-10"),
    ("3LYJfcfHPXYJreMsASk2jkn69LWEYKzexb", "Binance", BINANCE_URL, "2022-11-10"),
    ("3M219KR5vEneNb47ewrPfWyb5jQ2DjxRP6", "Binance", BINANCE_URL, "2022-11-10"),
    ("bc1qm34lsc65zpw79lxes69zkqmk6ee3ewf0j77s3h", "Binance", BINANCE_URL, "2022-11-10"),
]
MANUAL_CSV = "exchange_addresses_manual.csv"
MIN_OKX_ADDRESSES = 100      # stop if fewer OKX addresses are parsed (format not recognized)

BTC_RE = re.compile(r"^(?:[13][1-9A-HJ-NP-Za-km-z]{25,34}|bc1[02-9ac-hj-np-z]{8,87})$")
TOKEN_RE = re.compile(r"(?:bc1[02-9ac-hj-np-zA-HJ-NP-Z]{8,87}|\b[13][1-9A-HJ-NP-Za-km-z]{25,34}\b)")
COIN_COLS = ("coin", "currency", "symbol", "asset", "chain", "network", "token", "coin_name")

def norm(a):
    a = str(a).strip()
    return a.lower() if a.lower().startswith("bc1") else a

def from_table(df):
    cols = {str(c).lower().strip().strip("\ufeff").strip(): c for c in df.columns}
    coin = next((cols[c] for c in COIN_COLS if c in cols), None)
    addr_cols = [cols[c] for c in cols if "address" in c or c in ("addr", "wallet")]
    if coin is None or not addr_cols:
        return []                                  # skip if the currency cannot be determined
    sub = df[df[coin].astype(str).str.upper().str.strip().isin(["BTC", "BITCOIN"])]
    out = []
    for c in addr_cols:
        out += [norm(a) for a in sub[c].dropna().astype(str)]
    return [a for a in out if BTC_RE.match(a)]

def from_text(txt):
    lines = txt.splitlines()
    tagged = [l for l in lines if re.search(r"\bBTC\b|bitcoin", l, re.I)
              and not re.search(r"\b(?:BCH|BSV|BTG|SV|WBTC|BTCB|BEP\d*|ERC\d*|TRC\d*)\b|bitcoin[ -]?(?:cash|gold|sv)|wrapped|lightning", l, re.I)]
    use = tagged if tagged else lines
    return [norm(a) for l in use for a in TOKEN_RE.findall(l) if BTC_RE.match(norm(a))]

PREVIEWED = set()
def preview(name, b, n=3):
    key = name.split("/")[-1].split("_20")[0]
    if key in PREVIEWED: return
    PREVIEWED.add(key)
    head = b[:4000].decode("utf-8", "ignore").splitlines()[:n]
    print(f"   [{name}] first {n} lines:", *[l[:160] for l in head], sep="\n      ")

def parse_blob(name, b):
    name = name.lower()
    if not name.endswith(".zip"):
        preview(name, b)
    if name.endswith(".zip"):
        z = zipfile.ZipFile(io.BytesIO(b)); out = []
        for m in z.namelist():
            if not m.endswith("/"):
                out += parse_blob(m, z.read(m))
        return out
    if name.endswith((".xlsx", ".xls")):
        return [a for df in pd.read_excel(io.BytesIO(b), sheet_name=None, dtype=str).values() for a in from_table(df)]
    if name.endswith(".csv"):
        try:
            got = from_table(pd.read_csv(io.BytesIO(b), dtype=str, on_bad_lines="skip", engine="python"))
            if got:
                return got
        except Exception as e:
            print("   csv parsing failed, treated as text:", name, e)
    return from_text(b.decode("utf-8", "ignore"))

rows, src_log = [], []
for ex, sdate, url in EXCHANGE_SOURCES:
    try:
        fn = url.split("?")[0].split("/")[-1]
        if os.path.exists(fn):                       # local files take precedence
            blob = open(fn, "rb").read(); print(f"   reading local file {fn}")
        else:
            r = requests.get(url, timeout=300, headers={"User-Agent": "Mozilla/5.0 (research; qbtc)"})
            r.raise_for_status()
            blob = r.content
        got = sorted(set(parse_blob(fn, blob)))
        rows += [(a, ex, url, sdate) for a in got]
        src_log.append({"exchange": ex, "source_date": sdate, "url": url, "n_btc_addresses": len(got), "status": "ok"})
        print(f"{ex:9s} {sdate}  {len(got):>7,} BTC addresses")
    except Exception as e:
        src_log.append({"exchange": ex, "source_date": sdate, "url": url, "n_btc_addresses": 0, "status": f"failed: {e}"})
        print(f"{ex:9s} {sdate}  download/parsing failed: {e}")
rows += [(norm(a), ex, u, d) for a, ex, u, d in INLINE_ADDRESSES]
if os.path.exists(MANUAL_CSV):
    man = pd.read_csv(MANUAL_CSV, dtype=str)
    man["address"] = man.address.map(norm)
    man = man[man.address.map(lambda a: bool(BTC_RE.match(a)))]
    rows += list(man[["address", "exchange", "source_url", "source_date"]].itertuples(index=False, name=None))
    print(f"Manual list {MANUAL_CSV}: {len(man):,} addresses")

exa = (pd.DataFrame(rows, columns=["address", "exchange", "source_url", "source_date"])
         .drop_duplicates(["address", "exchange"]).reset_index(drop=True))
conflict = exa.groupby("address").exchange.nunique()
print("Addresses labeled for more than one exchange:", int((conflict > 1).sum()), "(labeled multi in 05)")
display(exa.groupby("exchange").address.nunique().rename("n_addresses").to_frame())
n_okx = exa.loc[exa.exchange == "OKX", "address"].nunique()
assert n_okx >= MIN_OKX_ADDRESSES, (
    f"Only {n_okx} BTC addresses parsed from OKX: check the first lines printed above to see the format, "
    "or download por_csv_*.zip by hand, upload to the Colab working directory and re-run this cell. Nothing has been spent yet.")
exa.to_csv("exchange_addresses.csv", index=False)
upload("exchange_addresses_raw", exa)

## Step 2 · Positive-control snapshots, `holder_snapshot_pc`

t−1 snapshots for FTX (2022-11-08) and Randstorm (2023-11-14). SQL of 02b with the risk-set condition replaced by balance ≥ 0.01 BTC (Randstorm-era keys are mostly dormant); `last_in_date` is kept so 05 can apply the main definition. Event and t−1 dates from `events` (event_set = positive_control). `USE_ALT_DATES=True` also builds snapshots for the alt_dates (FTX bankruptcy filing 11-11, Randstorm pre-announcement 10-10).

In [ ]:
USE_ALT_DATES = False

ev = q(f"SELECT event, event_set, event_date, alt_date, snap_date FROM `{DS}.events`")
for c in ["event_date", "alt_date", "snap_date"]:
    ev[c] = pd.to_datetime(ev[c])
pc = ev[ev.event_set == "positive_control"][["event", "event_date", "snap_date"]].copy()
if USE_ALT_DATES:
    alt = ev[(ev.event_set == "positive_control") & ev.alt_date.notna()].copy()
    alt = pd.DataFrame({"event": alt.event + "_alt", "event_date": alt.alt_date,
                        "snap_date": alt.alt_date - pd.Timedelta(days=1)})
    pc = pd.concat([pc, alt], ignore_index=True)
display(pc)

PC_SQL = ",\n    ".join(
    f"STRUCT('{r.event}' AS event, DATE '{r.event_date:%Y-%m-%d}' AS event_date, DATE '{r.snap_date:%Y-%m-%d}' AS snap_date)"
    for r in pc.itertuples())

SQL_PC = f'''
CREATE OR REPLACE TABLE `{DS}.holder_snapshot_pc`
CLUSTER BY event, grp
AS
WITH snaps AS (
  SELECT * FROM UNNEST([
    {PC_SQL}
  ])
),
held AS (
  SELECT s.event, s.event_date, s.snap_date, u.address, u.type, u.sats, u.created_date,
         (u.spent_date IS NULL OR u.spent_date > s.snap_date) AS alive
  FROM `{DS}.utxo_life` u
  CROSS JOIN snaps s
  WHERE u.type IN ('pubkey','pubkeyhash','witness_v0_keyhash','scripthash',
                   'witness_v0_scripthash','witness_v1_taproot')
    AND u.address IS NOT NULL
    AND u.created_date <= s.snap_date
    AND ( u.spent_date IS NULL OR u.spent_date > s.snap_date
          OR u.created_date > DATE_SUB(s.snap_date, INTERVAL 365 DAY) )
),
agg AS (
  SELECT
    a.event, a.event_date, a.snap_date, k.key_id,
    ANY_VALUE(k.key_kind)                          AS key_kind,
    SUM(IF(a.alive, a.sats, 0))                    AS sats,
    COUNTIF(a.alive)                               AS n_utxo,
    COUNT(DISTINCT IF(a.alive, a.address, NULL))   AS n_addr,
    MIN(IF(a.alive, a.created_date, NULL))         AS oldest_utxo_date,
    MAX(IF(a.alive, a.created_date, NULL))         AS newest_utxo_date,
    MAX(a.created_date)                            AS last_in_date,
    ARRAY_AGG(DISTINCT IF(a.alive, a.type, NULL) IGNORE NULLS)         AS types
  FROM held a
  JOIN `{DS}.address_key` k USING (address)
  WHERE k.key_id IS NOT NULL
  GROUP BY a.event, a.event_date, a.snap_date, k.key_id
  HAVING SUM(IF(a.alive, a.sats, 0)) >= {SATS_MIN}
)
SELECT
  g.*,
  e.first_exposed_date,
  e.first_exposed_src,
  CASE WHEN g.key_kind = 't'                       THEN 'T'
       WHEN e.first_exposed_date <= g.snap_date    THEN 'L'
       ELSE 'C' END AS grp
FROM agg g
LEFT JOIN `{DS}.key_exposure` e USING (key_id)
'''
build("holder_snapshot_pc", SQL_PC)
display(q(f'''
SELECT event, snap_date, grp, COUNT(*) AS n_keys, ROUND(SUM(sats)/1e8) AS btc,
       COUNTIF(last_in_date > DATE_SUB(snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)) AS n_keys_recent_in
FROM `{DS}.holder_snapshot_pc` GROUP BY 1, 2, 3 ORDER BY 1, 3'''))

## Step 3 · Common-input expansion → `exchange_keys`

One hop of the common-input heuristic, using only transactions whose labeled inputs belong to a single exchange and that have at most 1,000 inputs. Labels: `src = por` (official) or `cospend` (expanded); `y2_por` uses the official addresses only. Addresses are mapped to key_ids through `address_key`. `EXPAND_COSPEND=False` skips the expansion.

In [ ]:
need("exchange_addresses_raw")
if EXPAND_COSPEND:
    SQL_COSPEND = f'''
CREATE OR REPLACE TABLE `{DS}.exchange_cospend`
CLUSTER BY address
AS
WITH lab AS (
  SELECT address, ANY_VALUE(exchange) AS exchange, COUNT(DISTINCT exchange) AS n_ex
  FROM `{DS}.exchange_addresses_raw` GROUP BY address
),
lab_tx AS (
  SELECT u.spending_txid,
         ANY_VALUE(l.exchange) AS exchange,
         COUNT(DISTINCT l.exchange) AS n_ex,
         MAX(l.n_ex) AS max_ex_per_addr
  FROM `{DS}.utxo_life` u
  JOIN lab l USING (address)
  WHERE u.spending_txid IS NOT NULL
  GROUP BY u.spending_txid
),
co AS (   -- all inputs of the same transaction; n_in from a window function, to avoid referencing the CTE again (BigQuery would scan it twice)
  SELECT u.spending_txid, u.address, t.exchange,
         COUNT(*) OVER (PARTITION BY u.spending_txid) AS n_in
  FROM `{DS}.utxo_life` u
  JOIN lab_tx t USING (spending_txid)
  WHERE t.n_ex = 1 AND t.max_ex_per_addr = 1
)
SELECT co.address, ANY_VALUE(co.exchange) AS exchange, COUNT(DISTINCT co.exchange) AS n_ex,
       COUNT(DISTINCT co.spending_txid) AS n_tx
FROM co
WHERE co.n_in <= {COSPEND_MAX_IN}
  AND co.address IS NOT NULL
  AND co.address NOT IN (SELECT address FROM `{DS}.exchange_addresses_raw`)
GROUP BY co.address
'''
    build("exchange_cospend", SQL_COSPEND)

cospend_part = f'''
  UNION ALL
  SELECT address, exchange, 'cospend' AS src FROM `{DS}.exchange_cospend` WHERE n_ex = 1''' if EXPAND_COSPEND else ""
SQL_EXKEYS = f'''
CREATE OR REPLACE TABLE `{DS}.exchange_keys`
AS
WITH a AS (
  SELECT address, exchange, 'por' AS src FROM `{DS}.exchange_addresses_raw`{cospend_part}
)
SELECT k.key_id,
       IF(COUNT(DISTINCT a.exchange) = 1, ANY_VALUE(a.exchange), 'multi') AS exchange,
       IF(LOGICAL_OR(a.src = 'por'), 'por', 'cospend')                     AS src,
       COUNT(DISTINCT a.address)                                            AS n_addr
FROM a
JOIN `{DS}.address_key` k USING (address)
WHERE k.key_id IS NOT NULL
GROUP BY k.key_id
'''
build("exchange_keys", SQL_EXKEYS, force=True)
ex_summary = q(f'''
SELECT exchange, src, COUNT(*) AS n_keys, SUM(n_addr) AS n_addr
FROM `{DS}.exchange_keys` GROUP BY 1, 2 ORDER BY 1, 2''')
display(ex_summary)
onchain = q(f'''
SELECT r.exchange, COUNT(DISTINCT r.address) AS listed, COUNT(DISTINCT k.address) AS seen_on_chain
FROM `{DS}.exchange_addresses_raw` r LEFT JOIN `{DS}.address_key` k USING (address)
GROUP BY 1 ORDER BY 1''')
display(onchain)   # addresses never seen on chain (mostly same-format addresses of other chains) do not enter exchange_keys

## Step 4 · Key universe and exclusion flags

`flow_universe` = main-event snapshots (risk-set filter, without latest) ∪ placebo snapshots ∪ positive-control snapshots. `key_flags`: first-seen date (used for the Randstorm cohort), maximum receipts at one address, exchange label, `is_service`.

In [ ]:
need("holder_snapshot_pc", "exchange_keys")
SQL_UNIVERSE = f'''
CREATE OR REPLACE TABLE `{DS}.flow_universe`
AS
SELECT key_id,
       LOGICAL_OR(src = 'main')    AS in_main,
       LOGICAL_OR(src = 'placebo') AS in_placebo,
       LOGICAL_OR(src = 'pc')      AS in_pc
FROM (
  SELECT key_id, 'main' AS src FROM `{DS}.holder_snapshot`
  WHERE event != 'latest' AND sats >= {SATS_MIN}
    AND last_in_date > DATE_SUB(snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)
  UNION ALL
  SELECT key_id, 'placebo' FROM `{DS}.holder_snapshot_placebo`
  UNION ALL
  SELECT key_id, 'pc' FROM `{DS}.holder_snapshot_pc`
)
GROUP BY key_id
'''
build("flow_universe", SQL_UNIVERSE)

SQL_FLAGS = f'''
CREATE OR REPLACE TABLE `{DS}.key_flags`
AS
WITH ks AS (
  SELECT k.key_id,
         MIN(s.first_seen_date) AS first_seen_date,
         MAX(s.n_received)      AS max_addr_n_received,
         SUM(s.n_received)      AS key_n_received,
         COUNT(*)               AS n_addr_total
  FROM `{DS}.address_key` k
  JOIN `{DS}.flow_universe` u USING (key_id)
  JOIN `{DS}.address_stats` s USING (address)
  GROUP BY k.key_id
)
SELECT ks.*, u.in_main, u.in_placebo, u.in_pc,
       ek.exchange, ek.src AS exchange_src,
       ek.key_id IS NOT NULL AS is_exchange,
       (ek.key_id IS NOT NULL OR ks.max_addr_n_received > {SERVICE_MAX_RECV}) AS is_service
FROM ks
JOIN `{DS}.flow_universe` u USING (key_id)
LEFT JOIN `{DS}.exchange_keys` ek USING (key_id)
'''
build("key_flags", SQL_FLAGS)
display(q(f'''
SELECT in_main, in_placebo, in_pc, COUNT(*) AS n_keys, COUNTIF(is_exchange) AS n_exchange,
       COUNTIF(is_service) AS n_service
FROM `{DS}.key_flags` GROUP BY 1, 2, 3 ORDER BY 4 DESC'''))

## Step 5 · `flow_days` and `event_flow_coverage`

- Main and secondary events with snapshots, and positive controls: relative days −60…+60
- Placebo and negative-control dates: −14…+30
- Truncated at the last data day

`strict_ok`: event date + 30 + 30 ≤ last data day. `placebo_draws_strict` is re-drawn from the strict_ok placebo dates as in 03 (same seed, n_main dates without replacement); `placebo_draws` from 03 remains in use for y1 mid/loose, y2 and y3.

In [ ]:
snap_ev = q(f'''
SELECT DISTINCT event, event_date FROM `{DS}.holder_snapshot` WHERE event != 'latest'
UNION ALL SELECT DISTINCT event, event_date FROM `{DS}.holder_snapshot_pc`
UNION ALL SELECT DISTINCT event, event_date FROM `{DS}.holder_snapshot_placebo`''')
snap_ev["event_date"] = pd.to_datetime(snap_ev.event_date)
set_map = dict(zip(ev.event, ev.event_set))
def ev_set(e):
    if e.startswith("pl_"): return "placebo"
    if e.startswith("nq_"): return "nonquantum_news"
    return set_map.get(e.replace("_alt", ""), "unknown")
snap_ev["event_set"] = snap_ev.event.map(ev_set)
assert (snap_ev.event_set != "unknown").all(), snap_ev[snap_ev.event_set == "unknown"]

fd = []
for r in snap_ev.itertuples():
    lo, hi = WIDE_WIN if r.event_set in ("main", "secondary", "positive_control") else MAIN_WIN
    for k in range(lo, hi + 1):
        d = r.event_date + pd.Timedelta(days=k)
        if d <= DATA_END:
            fd.append((r.event, r.event_set, r.event_date, k, d, MAIN_WIN[0] <= k <= MAIN_WIN[1]))
flow_days = pd.DataFrame(fd, columns=["event", "event_set", "event_date", "rel_day", "date", "in_main_window"])
print(f"{len(snap_ev)} events/dates, {len(flow_days):,} rows, {flow_days.date.nunique()} distinct days:",
      flow_days.date.min().date(), "...", flow_days.date.max().date())
display(flow_days.groupby("event_set").agg(n_events=("event", "nunique"), n_days=("date", "nunique")))
upload("flow_days", flow_days)
FLOW_MIN, FLOW_MAX = flow_days.date.min(), flow_days.date.max()

cov = snap_ev.copy()
cov["main_last_day"] = cov.event_date + pd.Timedelta(days=MAIN_WIN[1])
cov["observed_main_days"] = [(min(m, DATA_END) - (e + pd.Timedelta(days=MAIN_WIN[0]))).days + 1
                             for e, m in zip(cov.event_date, cov.main_last_day)]
cov["main_window_complete"] = cov.main_last_day <= DATA_END
cov["strict_ok"] = cov.main_last_day + pd.Timedelta(days=DORMANT_DAYS) <= DATA_END
cov["data_end"] = DATA_END
print("Main window incomplete:", cov.loc[~cov.main_window_complete, "event"].tolist())
print("y1 strict observed for less than 30 days:", cov.loc[~cov.strict_ok, "event"].tolist())
assert cov.loc[cov.event_set.isin(["main", "positive_control"]), "strict_ok"].all(), "main events/positive controls are censored too; check DATA_END"
upload("event_flow_coverage", cov)
cov.to_csv("event_flow_coverage.csv", index=False)

# re-draw after filtering the placebo pool on strict_ok
pdays = q(f"SELECT kind, date FROM `{DS}.placebo_days`")
pdays["date"] = pd.to_datetime(pdays.date)
ok_dates = set(cov.loc[cov.strict_ok, "event_date"])
n_main = int((ev.event_set == "main").sum())
def make_draws(p, name, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame([(name, i + 1, j + 1, d) for i in range(N_DRAWS)
                        for j, d in enumerate(sorted(rng.choice(p, size=n_main, replace=False)))],
                       columns=["pool", "draw_id", "slot", "fake_date"])
    out["fake_date"] = pd.to_datetime(out.fake_date)
    return out
pools = {"main": "ri_pool", "sym30": "ri_pool_sym30"}
draws, pool_report = [], {}
for i, (pname, kind) in enumerate(pools.items()):
    full = sorted(pdays.loc[pdays.kind == kind, "date"])
    keep = [d for d in full if d in ok_dates]
    pool_report[pname] = {"before": len(full), "after": len(keep),
                          "dropped": [d.strftime("%Y-%m-%d") for d in full if d not in ok_dates]}
    print(f"pool {pname}: {len(full)} -> {len(keep)}, dropped {pool_report[pname]['dropped']}")
    draws.append(make_draws(keep, pname, SEED + 10 + i))
upload("placebo_draws_strict", pd.concat(draws, ignore_index=True))

## Step 6 · `flow_tx_inputs`

One scan of `utxo_life` outputs spent on `flow_days` dates, mapped to key_ids. Transactions with at least one input from a non-service risk-set key are kept with all their inputs (for change detection and proportional allocation). Set `INPUT_CHUNKS` to yearly ranges if the query exceeds resources.

In [ ]:
need("flow_days", "key_flags")
INPUT_CHUNKS = None       # e.g. [("2022-01-01","2023-12-31"), ("2024-01-01","2024-12-31"), ("2025-01-01","2025-12-31"), ("2026-01-01","2026-12-31")]

def inputs_sql(table, lo=None, hi=None):
    rng = f"WHERE date BETWEEN DATE '{lo}' AND DATE '{hi}'" if lo else ""
    return f'''
CREATE OR REPLACE TABLE `{DS}.{table}`
PARTITION BY spent_date
AS
WITH d AS (SELECT DISTINCT date FROM `{DS}.flow_days` {rng}),
rk AS (SELECT key_id FROM `{DS}.key_flags` WHERE NOT is_service),
inp AS (
  SELECT u.spending_txid, u.spent_date, u.address, u.type, u.sats, u.created_date,
         k.key_id
  FROM `{DS}.utxo_life` u
  JOIN d ON u.spent_date = d.date
  LEFT JOIN `{DS}.address_key` k ON k.address = u.address
)
SELECT i.*, rk.key_id IS NOT NULL AS is_risk_key
FROM inp i
LEFT JOIN rk ON rk.key_id = i.key_id
             AND i.type IN ('pubkey','pubkeyhash','witness_v0_keyhash','scripthash',
                            'witness_v0_scripthash','witness_v1_taproot')   -- same definition as the snapshots: bare multisig etc. are not part of key balances
WHERE TRUE
QUALIFY LOGICAL_OR(rk.key_id IS NOT NULL) OVER (PARTITION BY i.spending_txid)
'''

if INPUT_CHUNKS is None:
    build("flow_tx_inputs", inputs_sql("flow_tx_inputs"))
else:
    parts = []
    for j, (lo, hi) in enumerate(INPUT_CHUNKS):
        name = f"flow_tx_inputs_c{j:02d}"; parts.append(name)
        build(name, inputs_sql(name, lo, hi))
    need(*parts)
    build("flow_tx_inputs", f'''
CREATE OR REPLACE TABLE `{DS}.flow_tx_inputs` PARTITION BY spent_date AS
''' + "\nUNION ALL\n".join(f"SELECT * FROM `{DS}.{p}`" for p in parts))
    if table_rows("flow_tx_inputs"):
        for p in parts: client.delete_table(f"{DS}.{p}", not_found_ok=True)
display(q(f'''
SELECT COUNT(DISTINCT spending_txid) AS n_tx, COUNT(*) AS n_inputs, COUNTIF(is_risk_key) AS n_risk_inputs,
       COUNT(DISTINCT IF(is_risk_key, key_id, NULL)) AS n_risk_keys, ROUND(SUM(IF(is_risk_key, sats, 0))/1e8) AS risk_btc_spent
FROM `{DS}.flow_tx_inputs`'''))

## Step 7 · `flow_tx_outputs` and `dest_key_info`

Outputs are created in the block that spends the inputs (`created_date = spent_date`), so partition pruning on `created_date` limits the scan to 2022 onward. New address: `address_stats.first_seen_height = created_height`. New or exposed key: `dest_key_info` (key first appearance = minimum first-seen height over its addresses).

In [ ]:
need("flow_tx_inputs")
SQL_OUT = f'''
CREATE OR REPLACE TABLE `{DS}.flow_tx_outputs`
PARTITION BY created_date
AS
WITH tx AS (SELECT DISTINCT spending_txid AS txid, spent_date FROM `{DS}.flow_tx_inputs`)
SELECT u.txid, u.vout, u.created_date, u.created_height, u.sats, u.type, u.address,
       IF(u.spent_date <= DATE '{DATA_END:%Y-%m-%d}', u.spent_date, NULL) AS out_spent_date,   -- spends after the freeze date are treated as unspent
       k.key_id                    AS dest_key_id,
       k.key_kind                  AS dest_key_kind,
       s.first_seen_height         AS addr_first_seen_height
FROM `{DS}.utxo_life` u
JOIN tx ON tx.txid = u.txid AND tx.spent_date = u.created_date
LEFT JOIN `{DS}.address_key`   k ON k.address = u.address
LEFT JOIN `{DS}.address_stats` s ON s.address = u.address
WHERE u.created_date BETWEEN DATE '{FLOW_MIN:%Y-%m-%d}' AND DATE '{FLOW_MAX:%Y-%m-%d}'
'''
build("flow_tx_outputs", SQL_OUT)

SQL_DK = f'''
CREATE OR REPLACE TABLE `{DS}.dest_key_info`
AS
WITH dk AS (SELECT DISTINCT dest_key_id AS key_id FROM `{DS}.flow_tx_outputs` WHERE dest_key_id IS NOT NULL),
fs AS (
  SELECT k.key_id, MIN(s.first_seen_height) AS key_first_seen_height
  FROM `{DS}.address_key` k
  JOIN dk USING (key_id)
  JOIN `{DS}.address_stats` s USING (address)
  GROUP BY k.key_id
)
SELECT fs.key_id, fs.key_first_seen_height, e.first_exposed_date
FROM fs
LEFT JOIN `{DS}.key_exposure` e USING (key_id)
'''
build("dest_key_info", SQL_DK)

# check: every transaction has outputs, and total outputs <= total inputs (difference = fee; utxo_life has no OP_RETURN)
display(q(f'''
WITH i AS (SELECT spending_txid AS txid, SUM(sats) AS in_sats FROM `{DS}.flow_tx_inputs` GROUP BY 1),
     o AS (SELECT txid, SUM(sats) AS out_sats FROM `{DS}.flow_tx_outputs` GROUP BY 1)
SELECT COUNT(*) AS n_tx, COUNTIF(o.txid IS NULL) AS tx_without_outputs,
       COUNTIF(o.out_sats > i.in_sats) AS tx_out_gt_in,
       ROUND(SUM(i.in_sats - IFNULL(o.out_sats, 0)) / 1e8, 2) AS total_fee_btc
FROM i LEFT JOIN o USING (txid)'''))

## Step 8 · `flow_key_tx` and `key_day_flows`

Key share in a transaction: `share = key input value / total input value`; amount to each destination = share × non-change outputs to that destination. y1-strict transfers whose 30-day period passes the last data day are stored separately in `y1_strict_censored`. `newest_in_created` (latest creation date among the key's inputs in the transaction) lets 05 restrict to coins held before t−1 (`newest_in_created ≤ snap_date`) as a robustness check.

In [ ]:
need("flow_tx_outputs", "dest_key_info", "exchange_keys")
SQL_KTX = f'''
CREATE OR REPLACE TABLE `{DS}.flow_key_tx`
PARTITION BY date
AS
WITH tx_in AS (
  SELECT spending_txid AS txid, ANY_VALUE(spent_date) AS date, SUM(sats) AS in_sats, COUNT(*) AS n_in,
         ARRAY_AGG(DISTINCT key_id IGNORE NULLS) AS in_keys
  FROM `{DS}.flow_tx_inputs` GROUP BY txid
),
key_in AS (
  SELECT spending_txid AS txid, key_id, SUM(sats) AS key_in_sats, COUNT(*) AS n_utxo_in,
         MIN(created_date) AS oldest_in_created, MAX(created_date) AS newest_in_created
  FROM `{DS}.flow_tx_inputs` WHERE is_risk_key GROUP BY txid, key_id
),
o AS (
  SELECT o.txid, o.sats,
    IFNULL(o.dest_key_id IN UNNEST(t.in_keys), FALSE)                               AS is_change,
    ex.key_id IS NOT NULL                                                            AS to_ex,
    IFNULL(ex.src = 'por', FALSE)                                                    AS to_ex_por,
    (o.type IN ('pubkey', 'multisig', 'witness_v1_taproot')
       OR IFNULL(dk.first_exposed_date <= o.created_date, FALSE))                    AS exposed,
    o.dest_key_id IS NOT NULL                                                        AS keyed,
    IFNULL(dk.key_first_seen_height = o.created_height, FALSE)                       AS new_key,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)
      AND o.type IN ('pubkeyhash', 'witness_v0_keyhash', 'scripthash', 'witness_v0_scripthash')
      AND (o.out_spent_date IS NULL
           OR o.out_spent_date > DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY))  AS strict_ok,
    DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY) > DATE '{DATA_END:%Y-%m-%d}' AS censored
  FROM `{DS}.flow_tx_outputs` o
  JOIN tx_in t USING (txid)
  LEFT JOIN `{DS}.dest_key_info` dk ON dk.key_id = o.dest_key_id
  LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = o.dest_key_id
),
c AS (
  SELECT *,
    NOT is_change                                              AS nc,
    NOT is_change AND to_ex                                    AS c_y2,
    NOT is_change AND NOT to_ex AND exposed                    AS c_y0,
    NOT is_change AND NOT to_ex AND NOT exposed AND keyed      AS c_y1l,
    NOT is_change AND NOT to_ex AND NOT exposed AND NOT keyed  AS c_oth
  FROM o
),
tx_out AS (
  SELECT txid,
    SUM(sats)                                      AS out_sats,
    COUNT(*)                                       AS n_out,
    SUM(IF(is_change, sats, 0))                    AS change_sats,
    SUM(IF(nc, sats, 0))                           AS y3,
    SUM(IF(c_y2, sats, 0))                         AS y2,
    SUM(IF(nc AND to_ex_por, sats, 0))             AS y2_por,
    SUM(IF(c_y0, sats, 0))                         AS y0,
    SUM(IF(c_y1l, sats, 0))                        AS y1_loose,
    SUM(IF(c_y1l AND new_key, sats, 0))            AS y1_mid,
    SUM(IF(c_y1l AND strict_ok, sats, 0))          AS y1_strict,
    SUM(IF(c_y1l AND strict_ok AND censored, sats, 0)) AS y1_strict_censored,
    SUM(IF(c_oth, sats, 0))                        AS other
  FROM c GROUP BY txid
)
SELECT
  ki.key_id, t.date, ki.txid,
  ki.key_in_sats, ki.n_utxo_in, ki.oldest_in_created, ki.newest_in_created,
  t.in_sats, t.n_in, ARRAY_LENGTH(t.in_keys) AS n_in_keys, IFNULL(o.n_out, 0) AS n_out,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats)                          AS share,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.change_sats, 0)          AS change_sats,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * (t.in_sats - IFNULL(o.out_sats, 0)) AS fee_sats,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y3, 0)                   AS y3,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y2, 0)                   AS y2,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y2_por, 0)               AS y2_por,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y0, 0)                   AS y0,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_loose, 0)             AS y1_loose,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_mid, 0)               AS y1_mid,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_strict, 0)            AS y1_strict,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.y1_strict_censored, 0)   AS y1_strict_censored,
  SAFE_DIVIDE(ki.key_in_sats, t.in_sats) * IFNULL(o.other, 0)                AS other
FROM key_in ki
JOIN tx_in t USING (txid)
LEFT JOIN tx_out o USING (txid)          -- transactions with only OP_RETURN outputs have no output rows in utxo_life
'''
build("flow_key_tx", SQL_KTX)

SQL_KDF = f'''
CREATE OR REPLACE TABLE `{DS}.key_day_flows`
PARTITION BY date
AS
SELECT key_id, date,
  COUNT(*)                 AS n_tx,
  SUM(key_in_sats)         AS spent_sats,
  MIN(oldest_in_created)   AS oldest_in_created,
  MAX(newest_in_created)   AS newest_in_created,
  SUM(change_sats) AS change_sats, SUM(fee_sats) AS fee_sats,
  SUM(y3) AS y3, SUM(y2) AS y2, SUM(y2_por) AS y2_por, SUM(y0) AS y0,
  SUM(y1_loose) AS y1_loose, SUM(y1_mid) AS y1_mid, SUM(y1_strict) AS y1_strict,
  SUM(y1_strict_censored) AS y1_strict_censored, SUM(other) AS other
FROM `{DS}.flow_key_tx`
GROUP BY key_id, date
'''
build("key_day_flows", SQL_KDF)
print(table_info("flow_key_tx"), table_info("key_day_flows"))

## Checks

1. Identities: y3 = y2 + y0 + y1 loose + other; amount spent = change + fee + y3; y1 strict, y1 mid ≤ y1 loose; y2_por ≤ y2.
2. Risk-set key shares in one transaction sum to ≤ 1.
3. Total `spent_sats` in `key_day_flows` = total risk-set inputs in `flow_tx_inputs`.
4. Daily destination mix of main-event risk-set keys (BTC), with event days marked: `fig_flows.png`, `flows_daily_summary.csv`.
5. The 30 largest key-days, to spot missed exchanges or services.

In [ ]:
need("key_day_flows")
chk = q(f'''
SELECT
  ROUND(SUM(spent_sats)/1e8, 4)                                          AS spent_btc,
  ROUND(SUM(change_sats + fee_sats + y3 - spent_sats)/1e8, 6)            AS gap_spent_identity,
  ROUND(SUM(y2 + y0 + y1_loose + other - y3)/1e8, 6)                     AS gap_y3_identity,
  COUNTIF(y1_strict > y1_loose + 1) AS bad_strict, COUNTIF(y1_mid > y1_loose + 1) AS bad_mid,
  COUNTIF(y2_por > y2 + 1) AS bad_por, COUNTIF(fee_sats < -1) AS neg_fee
FROM `{DS}.key_day_flows`''')
display(chk)
share = q(f'''
SELECT COUNTIF(s > 1 + 1e-9) AS tx_share_gt_1, MAX(s) AS max_share_sum
FROM (SELECT txid, SUM(share) AS s FROM `{DS}.flow_key_tx` GROUP BY txid)''')
display(share)
tot = q(f'''
SELECT (SELECT SUM(sats) FROM `{DS}.flow_tx_inputs` WHERE is_risk_key) AS a,
       (SELECT SUM(spent_sats) FROM `{DS}.key_day_flows`) AS b''')
display(tot)
c = chk.iloc[0]
assert abs(c.gap_spent_identity) < 1e-3 * max(c.spent_btc, 1) and abs(c.gap_y3_identity) < 1e-3 * max(c.spent_btc, 1)
assert c.bad_strict == 0 and c.bad_mid == 0 and c.bad_por == 0 and c.neg_fee == 0
assert share.tx_share_gt_1.iloc[0] == 0 and int(tot.a.iloc[0]) == int(tot.b.iloc[0])
print("Identity and total checks passed")

In [ ]:
daily = q(f'''
SELECT f.date,
  ROUND(SUM(f.y3)/1e8, 2) AS y3, ROUND(SUM(f.y2)/1e8, 2) AS y2, ROUND(SUM(f.y0)/1e8, 2) AS y0,
  ROUND(SUM(f.y1_loose)/1e8, 2) AS y1_loose, ROUND(SUM(f.y1_mid)/1e8, 2) AS y1_mid,
  ROUND(SUM(f.y1_strict)/1e8, 2) AS y1_strict, ROUND(SUM(f.other)/1e8, 2) AS other,
  ROUND(SUM(f.change_sats)/1e8, 2) AS change, COUNT(DISTINCT f.key_id) AS n_keys
FROM `{DS}.key_day_flows` f
JOIN `{DS}.key_flags` k USING (key_id)
WHERE k.in_main
GROUP BY f.date ORDER BY f.date''')
daily["date"] = pd.to_datetime(daily.date)
daily.to_csv("flows_daily_summary.csv", index=False)

import matplotlib.pyplot as plt
main_ev = snap_ev[snap_ev.event_set.isin(["main", "secondary", "positive_control"])]
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
d = daily.set_index("date")
d[["y2", "y0", "y1_loose", "y1_strict"]].rolling(7, min_periods=1).mean().plot(ax=axes[0], lw=1)
axes[0].set_ylabel("BTC per day (7-day mean)"); axes[0].set_title("Outflow destinations, main-event risk-set keys")
(d[["y2", "y0", "y1_loose", "y1_strict"]].div(d.y3, axis=0)).rolling(7, min_periods=1).mean().plot(ax=axes[1], lw=1)
axes[1].set_ylabel("share of y3")
for ax in axes:
    for r in main_ev.itertuples():
        ax.axvline(r.event_date, color="grey", ls=":", lw=0.8)
plt.tight_layout(); plt.savefig("fig_flows.png", dpi=130); plt.show()

top = q(f'''
SELECT TO_HEX(f.key_id) AS key_id, f.date, f.n_tx, ROUND(f.spent_sats/1e8, 2) AS spent_btc,
       ROUND(f.y3/1e8, 2) AS y3, ROUND(f.y2/1e8, 2) AS y2, ROUND(f.y1_strict/1e8, 2) AS y1_strict,
       k.max_addr_n_received, k.key_n_received, k.first_seen_date
FROM `{DS}.key_day_flows` f JOIN `{DS}.key_flags` k USING (key_id)
ORDER BY f.spent_sats DESC LIMIT 30''')
display(top)

## Step 9 · y1-strict audit sample

200 y1-strict transactions from the main windows (−14…+30), ordered by a hash of txid, exported to `y1_audit_sample.csv`: transaction link, key share, and per output type, value, new-address flag and whether it was spent within 30 days. `manual_label`: `migration`, `payment`, `exchange_deposit`, `coinjoin`, `other`. The rule-based classification is in 22.

In [ ]:
audit = q(f'''
WITH cand AS (
  SELECT f.txid, f.key_id, f.date, f.share, f.key_in_sats, f.y1_strict, f.y3, f.n_in, f.n_in_keys, f.n_out
  FROM `{DS}.flow_key_tx` f
  WHERE f.y1_strict > 0
    AND f.date IN (SELECT date FROM `{DS}.flow_days` WHERE event_set = 'main' AND in_main_window)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY f.txid ORDER BY f.key_in_sats DESC) = 1
),
pick AS (
  SELECT * FROM cand ORDER BY FARM_FINGERPRINT(CONCAT(TO_HEX(txid), '{SEED}')) LIMIT {AUDIT_N}
)
SELECT TO_HEX(p.txid) AS txid, CONCAT('https://mempool.space/tx/', TO_HEX(p.txid)) AS url,
       TO_HEX(p.key_id) AS key_id, p.date, ROUND(p.share, 4) AS share,
       ROUND(p.key_in_sats/1e8, 8) AS key_in_btc, ROUND(p.y1_strict/1e8, 8) AS y1_strict_btc,
       ROUND(p.y3/1e8, 8) AS y3_btc, p.n_in, p.n_in_keys, p.n_out,
       STRING_AGG(FORMAT('%d:%s:%.8f:new_addr=%t:spent=%s', o.vout, o.type, o.sats/1e8,
                         IFNULL(o.addr_first_seen_height = o.created_height, FALSE),
                         IFNULL(CAST(o.out_spent_date AS STRING), '-')), ' | ' ORDER BY o.vout) AS outputs,
       '' AS manual_label, '' AS note
FROM pick p
JOIN `{DS}.flow_tx_outputs` o ON o.txid = p.txid AND o.created_date = p.date
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11
ORDER BY date''')
audit.to_csv("y1_audit_sample.csv", index=False)
print(len(audit), "transactions saved to y1_audit_sample.csv")
display(audit.head(10))

In [ ]:
for rec in JOBS[-6:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
TABLES_04 = ["holder_snapshot_pc", "exchange_addresses_raw", "exchange_cospend", "exchange_keys", "flow_universe",
             "key_flags", "flow_days", "event_flow_coverage", "placebo_draws_strict", "flow_tx_inputs",
             "flow_tx_outputs", "dest_key_info", "flow_key_tx", "key_day_flows"]
for t in TABLES_04:
    n = table_rows(t)
    print(f"{t:24s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"risk_min_btc": RISK_MIN_BTC, "risk_lookback_days": RISK_LOOKBACK_DAYS,
                     "service_max_recv": SERVICE_MAX_RECV, "main_win": MAIN_WIN, "wide_win": WIDE_WIN,
                     "dormant_days": DORMANT_DAYS, "expand_cospend": EXPAND_COSPEND,
                     "cospend_max_in": COSPEND_MAX_IN, "data_end": str(DATA_END.date())},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in TABLES_04 if table_rows(t)]}
for k, v in [("exchange_sources", "src_log"), ("placebo_pools_strict", "pool_report")]:
    try: report[k] = eval(v)
    except NameError: pass
for k, v in [("exchange_keys", "ex_summary"), ("checks", "chk")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open("build_report_04.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_04.json", "exchange_addresses.csv", "event_flow_coverage.csv",
               "flows_daily_summary.csv", "fig_flows.png", "y1_audit_sample.csv"]:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("Report saved to build_report_04.json")